# Apache Polaris — Production Scenario: DataCorp Data Platform

**Date:** 2026-05-15

## Scenario Overview
Enterprise data platform with 5 catalogs, 10 principal roles, 12 principals, nested namespaces, and views as security/masking layer.

## Architecture
```
raw-prod       → landing zone (web, mobile, payment, system)
analytics-prod → business analytics (staging, mart, reporting)
ml-prod        → ML platform (features, models, experiments, monitoring)
finance-prod   → financial data PCI/SOX compliant (ledger, reporting, audit)
platform-prod  → infrastructure observability (metrics, incidents, governance)
```

## Key Production Patterns
- View as data masking (finance PII)
- Model promotion gate (ml-engineer vs ml-lead)
- Service account least privilege (one per pipeline)
- Data steward (metadata only, no data access)
- Audit service (properties only, all catalogs)
- Finance isolation (analyst sees views only, not raw tables)

In [57]:
# ============================================================
# Setup — DATACORP-PROD Realm
# ============================================================
import requests
import json
import pandas as pd
from IPython.display import display

POLARIS_URL    = 'http://192.168.139.2:8181'
REALM          = 'DATACORP-PROD'          # ← new realm
ROOT_CLIENT_ID = 'datacorp-root'           # ← new root
ROOT_SECRET    = 'datacorp-secret'         # ← new secret

def get_token(client_id=ROOT_CLIENT_ID, client_secret=ROOT_SECRET):
    r = requests.post(
        f'{POLARIS_URL}/api/catalog/v1/oauth/tokens',
        headers={'Polaris-Realm': REALM},   # ← realm header
        data={
            'grant_type': 'client_credentials',
            'client_id': client_id,
            'client_secret': client_secret,
            'scope': 'PRINCIPAL_ROLE:ALL'
        }
    )
    return r.json()['access_token']

def h(token):
    return {
        'Authorization': f'Bearer {token}',
        'Polaris-Realm': REALM,
        'Content-Type': 'application/json'
    }

TOKEN = get_token()
BASE  = f'{POLARIS_URL}/api/management/v1'
CAT   = f'{POLARIS_URL}/api/catalog/v1'
print(f'✅ Connected to realm: {REALM}')
print(f'   root: {ROOT_CLIENT_ID}')

✅ Connected to realm: DATACORP-PROD
   root: datacorp-root


## Step 1 — Principal Roles
조직 구조를 반영한 10개의 Principal Role 생성.
Principal Role은 항상 먼저 설계 — 사람/서비스보다 역할이 안정적.

In [58]:
print('=== Step 1: Create Principal Roles ===')

principal_roles = [
    ('data-engineer',   'Build and maintain data pipelines'),
    ('data-steward',    'Metadata management only, no data access'),
    ('ml-engineer',     'ML development, no production model write'),
    ('ml-lead',         'ML development + production model promotion'),
    ('analyst',         'Business analytics read access'),
    ('finance-analyst', 'Finance reporting views only, no PII'),
    ('finance-admin',   'Full finance access including PII ledger'),
    ('platform-ops',    'Infrastructure management'),
    ('security-auditor','Read properties only across all catalogs'),
    ('pipeline-reader', 'CI/CD schema validation only'),
]

for name, desc in principal_roles:
    post(f'{BASE}/principal-roles',
         {'principalRole': {'name': name, 'properties': {'description': desc}}},
         f'principal-role: {name}')

=== Step 1: Create Principal Roles ===
  ⚠️  [409] principal-role: data-engineer
       Cannot create PrincipalRole data-engineer. PrincipalRole already exists or resolution failed
  ⚠️  [409] principal-role: data-steward
       Cannot create PrincipalRole data-steward. PrincipalRole already exists or resolution failed
  ⚠️  [409] principal-role: ml-engineer
       Cannot create PrincipalRole ml-engineer. PrincipalRole already exists or resolution failed
  ⚠️  [409] principal-role: ml-lead
       Cannot create PrincipalRole ml-lead. PrincipalRole already exists or resolution failed
  ⚠️  [409] principal-role: analyst
       Cannot create PrincipalRole analyst. PrincipalRole already exists or resolution failed
  ⚠️  [409] principal-role: finance-analyst
       Cannot create PrincipalRole finance-analyst. PrincipalRole already exists or resolution failed
  ⚠️  [409] principal-role: finance-admin
       Cannot create PrincipalRole finance-admin. PrincipalRole already exists or resolution 

## Step 2 — Principals
12개 Principal 생성 — 사람 6명, 서비스 계정 6개.
서비스 계정은 파이프라인별 1개 (최소 권한 원칙).

In [59]:
print('=== Step 2: Create Principals ===')

principals = [
    # Humans
    ('alice',            ['data-engineer', 'data-steward']),   # tech lead — dual role
    ('bob',              ['analyst']),
    ('carol',            ['ml-lead']),
    ('dave',             ['finance-admin']),
    ('eve',              ['platform-ops', 'security-auditor']),# dual role
    ('frank',            ['finance-analyst']),
    # Service accounts
    ('spark-ingestion',  ['data-engineer']),  # raw write only
    ('spark-transform',  ['data-engineer']),  # raw read, analytics.staging write
    ('trino-reporting',  ['analyst']),         # reporting read only
    ('mlflow-service',   ['ml-engineer']),
    ('grafana-service',  ['platform-ops']),
    ('audit-service',    ['security-auditor']),
]

CREDENTIALS = {}

for pname, roles in principals:
    # Create principal
    r = requests.post(f'{BASE}/principals',
                      headers=h(TOKEN),
                      json={'name': pname, 'type': 'SERVICE'})
    if r.status_code == 201:
        creds = r.json().get('credentials', {})
        CREDENTIALS[pname] = (creds.get('clientId'), creds.get('clientSecret'))
        print(f'  ✅ [{r.status_code}] principal: {pname} (clientId: {creds.get("clientId", "?")})')
    else:
        print(f'  ⚠️  [{r.status_code}] principal: {pname}')

    # Assign principal roles
    for role in roles:
        put(f'{BASE}/principals/{pname}/principal-roles',
            {'principalRole': {'name': role}},
            f'  assign {pname} → {role}')

print(f'\n✅ {len(CREDENTIALS)} principals created')
print('⚠️  Save CREDENTIALS dict — clientSecrets shown only once')

=== Step 2: Create Principals ===
  ⚠️  [409] principal: alice
  ⚠️  [500]   assign alice → data-engineer
       Failed to write to grant records due to Failed due to 'ERROR: duplicate key value violates unique constraint "grant_records_pkey"
  Detail: Key (realm_id, securable_catalog_id, securable_id, grantee_catalog_id, grantee_id, privilege_code)=(DATACORP-PROD, 0, 6167224473334372285, 0, 8146456055196773787, 4) already exists.' (error code 0, sql-state '23505'), after 1 attempts and 5000 milliseconds
  ⚠️  [500]   assign alice → data-steward
       Failed to write to grant records due to Failed due to 'ERROR: duplicate key value violates unique constraint "grant_records_pkey"
  Detail: Key (realm_id, securable_catalog_id, securable_id, grantee_catalog_id, grantee_id, privilege_code)=(DATACORP-PROD, 0, 4259295114661435971, 0, 8146456055196773787, 4) already exists.' (error code 0, sql-state '23505'), after 1 attempts and 5000 milliseconds
  ⚠️  [409] principal: bob
  ⚠️  [500]   ass

## Step 3 — Catalogs
5개 카탈로그 생성 — 각 도메인별 완전한 격리.

In [60]:
print('=== Step 3: Create Catalogs ===')

MINIO_ENDPOINT = 'http://benchmarks-minio.datahub-hynix.svc.cluster.local:9000'
BUCKET         = 's3a://data-catalog-bucket'

catalogs = [
    ('raw-prod',       f'{BUCKET}/datacorp/raw/',       'Landing zone for raw ingestion data'),
    ('analytics-prod', f'{BUCKET}/datacorp/analytics/', 'Business analytics and reporting'),
    ('ml-prod',        f'{BUCKET}/datacorp/ml/',        'ML platform: features, models, experiments'),
    ('finance-prod',   f'{BUCKET}/datacorp/finance/',   'Financial data — PCI/SOX compliant'),
    ('platform-prod',  f'{BUCKET}/datacorp/platform/',  'Infrastructure and observability'),
]

for cname, location, desc in catalogs:
    post(f'{BASE}/catalogs', {
        'catalog': {
            'name': cname,
            'type': 'INTERNAL',
            'properties': {
                'default-base-location': location,
                'description': desc
            },
            'storageConfigInfo': {
                'storageType': 'S3',
                'allowedLocations': [f'{BUCKET}/datacorp/'],
                'roleArn': 'arn:aws:iam::123456789012:role/polaris-role',
                'pathStyleAccess': True
            }
        }
    }, f'catalog: {cname}')

=== Step 3: Create Catalogs ===
  ⚠️  [409] catalog: raw-prod
       Cannot create Catalog raw-prod. Catalog already exists or resolution failed
  ⚠️  [409] catalog: analytics-prod
       Cannot create Catalog analytics-prod. Catalog already exists or resolution failed
  ⚠️  [409] catalog: ml-prod
       Cannot create Catalog ml-prod. Catalog already exists or resolution failed
  ⚠️  [409] catalog: finance-prod
       Cannot create Catalog finance-prod. Catalog already exists or resolution failed
  ⚠️  [409] catalog: platform-prod
       Cannot create Catalog platform-prod. Catalog already exists or resolution failed


## Step 4 — Catalog Roles + Privileges
각 카탈로그별 역할 생성 및 권한 부여.

In [61]:
print('=== Step 4: Create Catalog Roles and Grant Privileges ===')

# Helper
def create_catalog_role(catalog, role):
    post(f'{BASE}/catalogs/{catalog}/catalog-roles',
         {'catalogRole': {'name': role}},
         f'catalog-role: {catalog}/{role}')

def grant(catalog, role, grant_body, label):
    put(f'{BASE}/catalogs/{catalog}/catalog-roles/{role}/grants',
        {'grant': grant_body}, f'  grant {label} → {role}@{catalog}')

# ── raw-prod ────────────────────────────────────────────────
print('\n── raw-prod ──')
for role in ['rw-role', 'ro-role', 'meta-role', 'prop-role', 'write-web-role', 'write-payment-role']:
    create_catalog_role('raw-prod', role)

# rw-role: full catalog access (data-engineer)
grant('raw-prod', 'rw-role', {'type': 'catalog', 'privilege': 'CATALOG_MANAGE_CONTENT'}, 'MANAGE_CONTENT')
grant('raw-prod', 'rw-role', {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}, 'READ_PROP')

# ro-role: read only (ml-engineer, spark-transform)
grant('raw-prod', 'ro-role', {'type': 'catalog', 'privilege': 'TABLE_READ_DATA'}, 'TABLE_READ')
grant('raw-prod', 'ro-role', {'type': 'catalog', 'privilege': 'NAMESPACE_LIST'}, 'NS_LIST')
grant('raw-prod', 'ro-role', {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}, 'READ_PROP')

# meta-role: metadata only (data-steward)
grant('raw-prod', 'meta-role', {'type': 'catalog', 'privilege': 'CATALOG_MANAGE_METADATA'}, 'MANAGE_META')
grant('raw-prod', 'meta-role', {'type': 'catalog', 'privilege': 'NAMESPACE_LIST'}, 'NS_LIST')

# prop-role: properties only (security-auditor, platform-ops)
grant('raw-prod', 'prop-role', {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}, 'READ_PROP')

# write-web-role: write web.* and mobile.* namespaces (spark-ingestion)
grant('raw-prod', 'write-web-role', {'type': 'namespace', 'namespace': ['web'], 'privilege': 'NAMESPACE_FULL_METADATA'}, 'NS_FULL web')
grant('raw-prod', 'write-web-role', {'type': 'namespace', 'namespace': ['mobile'], 'privilege': 'NAMESPACE_FULL_METADATA'}, 'NS_FULL mobile')
grant('raw-prod', 'write-web-role', {'type': 'catalog', 'privilege': 'TABLE_WRITE_DATA'}, 'TABLE_WRITE')

# write-payment-role: write payment.* namespace (spark-ingestion)
grant('raw-prod', 'write-payment-role', {'type': 'namespace', 'namespace': ['payment'], 'privilege': 'NAMESPACE_FULL_METADATA'}, 'NS_FULL payment')
grant('raw-prod', 'write-payment-role', {'type': 'catalog', 'privilege': 'TABLE_WRITE_DATA'}, 'TABLE_WRITE')

# ── analytics-prod ──────────────────────────────────────────
print('\n── analytics-prod ──')
for role in ['rw-role', 'ro-reporting-role', 'ro-staging-role', 'write-staging-role', 'meta-role', 'prop-role']:
    create_catalog_role('analytics-prod', role)

grant('analytics-prod', 'rw-role', {'type': 'catalog', 'privilege': 'CATALOG_MANAGE_CONTENT'}, 'MANAGE_CONTENT')
grant('analytics-prod', 'rw-role', {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}, 'READ_PROP')

# ro-reporting-role: read reporting.* only (analyst, trino-reporting, finance reporting)
grant('analytics-prod', 'ro-reporting-role', {'type': 'namespace', 'namespace': ['reporting'], 'privilege': 'NAMESPACE_FULL_METADATA'}, 'NS_FULL reporting')
grant('analytics-prod', 'ro-reporting-role', {'type': 'namespace', 'namespace': ['reporting'], 'privilege': 'TABLE_READ_DATA'}, 'TABLE_READ reporting')

# ro-staging-role: read staging.* only (ml-engineer, mlflow)
grant('analytics-prod', 'ro-staging-role', {'type': 'namespace', 'namespace': ['staging'], 'privilege': 'NAMESPACE_FULL_METADATA'}, 'NS_FULL staging')
grant('analytics-prod', 'ro-staging-role', {'type': 'namespace', 'namespace': ['staging'], 'privilege': 'TABLE_READ_DATA'}, 'TABLE_READ staging')

# write-staging-role: write staging.* only (spark-transform)
grant('analytics-prod', 'write-staging-role', {'type': 'namespace', 'namespace': ['staging'], 'privilege': 'NAMESPACE_FULL_METADATA'}, 'NS_FULL staging')
grant('analytics-prod', 'write-staging-role', {'type': 'catalog', 'privilege': 'TABLE_WRITE_DATA'}, 'TABLE_WRITE')

grant('analytics-prod', 'meta-role', {'type': 'catalog', 'privilege': 'CATALOG_MANAGE_METADATA'}, 'MANAGE_META')
grant('analytics-prod', 'meta-role', {'type': 'catalog', 'privilege': 'NAMESPACE_LIST'}, 'NS_LIST')
grant('analytics-prod', 'prop-role', {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}, 'READ_PROP')

# ── ml-prod ─────────────────────────────────────────────────
print('\n── ml-prod ──')
for role in ['full-role', 'engineer-role', 'prop-role']:
    create_catalog_role('ml-prod', role)

# full-role: full access (ml-lead, mlflow-service)
grant('ml-prod', 'full-role', {'type': 'catalog', 'privilege': 'CATALOG_MANAGE_CONTENT'}, 'MANAGE_CONTENT')
grant('ml-prod', 'full-role', {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}, 'READ_PROP')

# engineer-role: full EXCEPT models.production write (ml-engineer)
grant('ml-prod', 'engineer-role', {'type': 'namespace', 'namespace': ['features'], 'privilege': 'NAMESPACE_FULL_METADATA'}, 'NS_FULL features')
grant('ml-prod', 'engineer-role', {'type': 'namespace', 'namespace': ['experiments'], 'privilege': 'NAMESPACE_FULL_METADATA'}, 'NS_FULL experiments')
grant('ml-prod', 'engineer-role', {'type': 'namespace', 'namespace': ['monitoring'], 'privilege': 'NAMESPACE_FULL_METADATA'}, 'NS_FULL monitoring')
grant('ml-prod', 'engineer-role', {'type': 'catalog', 'privilege': 'TABLE_READ_DATA'}, 'TABLE_READ all')
grant('ml-prod', 'engineer-role', {'type': 'catalog', 'privilege': 'TABLE_WRITE_DATA'}, 'TABLE_WRITE all')
# models.production: READ only for engineer (no write)
grant('ml-prod', 'engineer-role', {'type': 'namespace', 'namespace': ['models'], 'privilege': 'NAMESPACE_FULL_METADATA'}, 'NS_FULL models')

grant('ml-prod', 'prop-role', {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}, 'READ_PROP')

# ── finance-prod ────────────────────────────────────────────
print('\n── finance-prod ──')
for role in ['admin-role', 'analyst-role', 'prop-role']:
    create_catalog_role('finance-prod', role)

# admin-role: full access including PII ledger
grant('finance-prod', 'admin-role', {'type': 'catalog', 'privilege': 'CATALOG_MANAGE_CONTENT'}, 'MANAGE_CONTENT')
grant('finance-prod', 'admin-role', {'type': 'catalog', 'privilege': 'CATALOG_MANAGE_ACCESS'}, 'MANAGE_ACCESS')
grant('finance-prod', 'admin-role', {'type': 'catalog', 'privilege': 'TABLE_READ_DATA'}, 'TABLE_READ')
grant('finance-prod', 'admin-role', {'type': 'catalog', 'privilege': 'TABLE_WRITE_DATA'}, 'TABLE_WRITE')

# analyst-role: reporting.* only (NO ledger access — PII protection)
grant('finance-prod', 'analyst-role', {'type': 'namespace', 'namespace': ['reporting'], 'privilege': 'NAMESPACE_FULL_METADATA'}, 'NS_FULL reporting')
grant('finance-prod', 'analyst-role', {'type': 'namespace', 'namespace': ['reporting'], 'privilege': 'TABLE_READ_DATA'}, 'TABLE_READ reporting')

grant('finance-prod', 'prop-role', {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}, 'READ_PROP')

# ── platform-prod ────────────────────────────────────────────
print('\n── platform-prod ──')
for role in ['full-role', 'ro-role', 'prop-role']:
    create_catalog_role('platform-prod', role)

grant('platform-prod', 'full-role', {'type': 'catalog', 'privilege': 'CATALOG_MANAGE_CONTENT'}, 'MANAGE_CONTENT')
grant('platform-prod', 'full-role', {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}, 'READ_PROP')
grant('platform-prod', 'full-role', {'type': 'catalog', 'privilege': 'TABLE_READ_DATA'}, 'TABLE_READ')
grant('platform-prod', 'full-role', {'type': 'catalog', 'privilege': 'TABLE_WRITE_DATA'}, 'TABLE_WRITE')

grant('platform-prod', 'ro-role', {'type': 'catalog', 'privilege': 'TABLE_READ_DATA'}, 'TABLE_READ')
grant('platform-prod', 'ro-role', {'type': 'catalog', 'privilege': 'NAMESPACE_LIST'}, 'NS_LIST')

grant('platform-prod', 'prop-role', {'type': 'catalog', 'privilege': 'CATALOG_READ_PROPERTIES'}, 'READ_PROP')

print('\n✅ Step 4 complete')

=== Step 4: Create Catalog Roles and Grant Privileges ===

── raw-prod ──
  ⚠️  [409] catalog-role: raw-prod/rw-role
       Cannot create CatalogRole rw-role in raw-prod. CatalogRole already exists or resolution failed
  ⚠️  [409] catalog-role: raw-prod/ro-role
       Cannot create CatalogRole ro-role in raw-prod. CatalogRole already exists or resolution failed
  ⚠️  [409] catalog-role: raw-prod/meta-role
       Cannot create CatalogRole meta-role in raw-prod. CatalogRole already exists or resolution failed
  ⚠️  [409] catalog-role: raw-prod/prop-role
       Cannot create CatalogRole prop-role in raw-prod. CatalogRole already exists or resolution failed
  ⚠️  [409] catalog-role: raw-prod/write-web-role
       Cannot create CatalogRole write-web-role in raw-prod. CatalogRole already exists or resolution failed
  ⚠️  [409] catalog-role: raw-prod/write-payment-role
       Cannot create CatalogRole write-payment-role in raw-prod. CatalogRole already exists or resolution failed
  ⚠️  [500] 

## Step 5 — Assign Catalog Roles to Principal Roles
Principal Role과 Catalog Role 연결 — 권한 체계의 핵심.

In [62]:
print('=== Step 5: Assign Catalog Roles to Principal Roles ===')

def assign_cr(principal_role, catalog, catalog_role):
    put(f'{BASE}/principal-roles/{principal_role}/catalog-roles/{catalog}',
        {'catalogRole': {'name': catalog_role}},
        f'{principal_role} → {catalog_role}@{catalog}')

# data-engineer: raw FULL, analytics FULL, ml READ, platform PROP
assign_cr('data-engineer', 'raw-prod',       'rw-role')
assign_cr('data-engineer', 'analytics-prod', 'rw-role')
assign_cr('data-engineer', 'ml-prod',        'prop-role')
assign_cr('data-engineer', 'platform-prod',  'prop-role')

# data-steward: meta only on raw + analytics
assign_cr('data-steward', 'raw-prod',       'meta-role')
assign_cr('data-steward', 'analytics-prod', 'meta-role')
assign_cr('data-steward', 'ml-prod',        'prop-role')
assign_cr('data-steward', 'platform-prod',  'prop-role')

# ml-engineer: raw READ, analytics staging READ, ml engineer-role
assign_cr('ml-engineer', 'raw-prod',       'ro-role')
assign_cr('ml-engineer', 'analytics-prod', 'ro-staging-role')
assign_cr('ml-engineer', 'ml-prod',        'engineer-role')
assign_cr('ml-engineer', 'platform-prod',  'prop-role')

# ml-lead: same as ml-engineer but full ml-prod access (model promotion)
assign_cr('ml-lead', 'raw-prod',       'ro-role')
assign_cr('ml-lead', 'analytics-prod', 'ro-staging-role')
assign_cr('ml-lead', 'ml-prod',        'full-role')
assign_cr('ml-lead', 'platform-prod',  'prop-role')

# analyst: analytics reporting READ, finance reporting READ
assign_cr('analyst', 'analytics-prod', 'ro-reporting-role')

# finance-analyst: finance reporting only (NO ledger/audit)
assign_cr('finance-analyst', 'analytics-prod', 'ro-reporting-role')
assign_cr('finance-analyst', 'finance-prod',   'analyst-role')

# finance-admin: full finance, platform prop
assign_cr('finance-admin', 'finance-prod',  'admin-role')
assign_cr('finance-admin', 'platform-prod', 'prop-role')

# platform-ops: platform FULL, all others PROP
assign_cr('platform-ops', 'raw-prod',       'prop-role')
assign_cr('platform-ops', 'analytics-prod', 'prop-role')
assign_cr('platform-ops', 'ml-prod',        'prop-role')
assign_cr('platform-ops', 'finance-prod',   'prop-role')
assign_cr('platform-ops', 'platform-prod',  'full-role')

# security-auditor: READ_PROP only on all catalogs
for cat in ['raw-prod', 'analytics-prod', 'ml-prod', 'finance-prod', 'platform-prod']:
    assign_cr('security-auditor', cat, 'prop-role')

# pipeline-reader: TABLE_READ_PROPERTIES raw + analytics (CI/CD validation)
assign_cr('pipeline-reader', 'raw-prod',       'prop-role')
assign_cr('pipeline-reader', 'analytics-prod', 'prop-role')

print('\n✅ Step 5 complete')

=== Step 5: Assign Catalog Roles to Principal Roles ===
  ⚠️  [500] data-engineer → rw-role@raw-prod
       Failed to write to grant records due to Failed due to 'ERROR: duplicate key value violates unique constraint "grant_records_pkey"
  Detail: Key (realm_id, securable_catalog_id, securable_id, grantee_catalog_id, grantee_id, privilege_code)=(DATACORP-PROD, 3161657845827158016, 8729824574806383020, 0, 6167224473334372285, 3) already exists.' (error code 0, sql-state '23505'), after 1 attempts and 5000 milliseconds
  ⚠️  [404] data-engineer → rw-role@analytics-prod
       Entity entityNames:[rw-role];lastEntityType:CATALOG_ROLE;isOptional:false not found when trying to assign analytics-prod.rw-role to data-engineer
  ⚠️  [500] data-engineer → prop-role@ml-prod
       Failed to write to grant records due to Failed due to 'ERROR: duplicate key value violates unique constraint "grant_records_pkey"
  Detail: Key (realm_id, securable_catalog_id, securable_id, grantee_catalog_id, grantee_i

## Step 6 — Namespaces (Nested)
중첩 네임스페이스 생성 — 도메인별 계층 구조.

In [63]:
print('=== Step 6: Create Namespaces ===')

def create_ns(catalog, namespace, location_suffix):
    ns_parts = namespace if isinstance(namespace, list) else [namespace]
    r = requests.post(
        f'{CAT}/{catalog}/namespaces',
        headers=h(TOKEN),
        json={
            'namespace': ns_parts,
            'properties': {
                'location': f's3a://data-catalog-bucket/datacorp/{location_suffix}'
            }
        }
    )
    ns_str = '.'.join(ns_parts)
    icon = '✅' if r.status_code in [200, 201] else '⚠️ '
    print(f'  {icon} [{r.status_code}] {catalog}/{ns_str}')

# raw-prod namespaces
print('\n── raw-prod ──')
create_ns('raw-prod', ['web'],                  'raw/web')
create_ns('raw-prod', ['web', 'clickstream'],   'raw/web/clickstream')
create_ns('raw-prod', ['web', 'pageviews'],     'raw/web/pageviews')
create_ns('raw-prod', ['mobile'],               'raw/mobile')
create_ns('raw-prod', ['mobile', 'events'],     'raw/mobile/events')
create_ns('raw-prod', ['mobile', 'crashes'],    'raw/mobile/crashes')
create_ns('raw-prod', ['payment'],              'raw/payment')
create_ns('raw-prod', ['payment', 'transactions'], 'raw/payment/transactions')
create_ns('raw-prod', ['payment', 'refunds'],   'raw/payment/refunds')
create_ns('raw-prod', ['system'],               'raw/system')
create_ns('raw-prod', ['system', 'logs'],       'raw/system/logs')

# analytics-prod namespaces
print('\n── analytics-prod ──')
create_ns('analytics-prod', ['staging'],             'analytics/staging')
create_ns('analytics-prod', ['staging', 'web'],      'analytics/staging/web')
create_ns('analytics-prod', ['staging', 'payment'],  'analytics/staging/payment')
create_ns('analytics-prod', ['mart'],                'analytics/mart')
create_ns('analytics-prod', ['mart', 'user'],        'analytics/mart/user')
create_ns('analytics-prod', ['mart', 'revenue'],     'analytics/mart/revenue')
create_ns('analytics-prod', ['reporting'],           'analytics/reporting')
create_ns('analytics-prod', ['reporting', 'daily'],  'analytics/reporting/daily')
create_ns('analytics-prod', ['reporting', 'monthly'],'analytics/reporting/monthly')

# ml-prod namespaces
print('\n── ml-prod ──')
create_ns('ml-prod', ['features'],              'ml/features')
create_ns('ml-prod', ['features', 'user'],      'ml/features/user')
create_ns('ml-prod', ['features', 'product'],   'ml/features/product')
create_ns('ml-prod', ['models'],                'ml/models')
create_ns('ml-prod', ['models', 'experimental'],'ml/models/experimental')
create_ns('ml-prod', ['models', 'production'],  'ml/models/production')
create_ns('ml-prod', ['experiments'],           'ml/experiments')
create_ns('ml-prod', ['experiments', 'runs'],   'ml/experiments/runs')
create_ns('ml-prod', ['monitoring'],            'ml/monitoring')
create_ns('ml-prod', ['monitoring', 'drift'],   'ml/monitoring/drift')

# finance-prod namespaces
print('\n── finance-prod ──')
create_ns('finance-prod', ['ledger'],               'finance/ledger')
create_ns('finance-prod', ['ledger', 'accounts'],   'finance/ledger/accounts')
create_ns('finance-prod', ['ledger', 'entries'],    'finance/ledger/entries')
create_ns('finance-prod', ['reporting'],            'finance/reporting')
create_ns('finance-prod', ['reporting', 'pnl'],     'finance/reporting/pnl')
create_ns('finance-prod', ['reporting', 'balance'], 'finance/reporting/balance')
create_ns('finance-prod', ['audit'],                'finance/audit')
create_ns('finance-prod', ['audit', 'access'],      'finance/audit/access')
create_ns('finance-prod', ['audit', 'changes'],     'finance/audit/changes')

# platform-prod namespaces
print('\n── platform-prod ──')
create_ns('platform-prod', ['metrics'],              'platform/metrics')
create_ns('platform-prod', ['metrics', 'infra'],     'platform/metrics/infra')
create_ns('platform-prod', ['metrics', 'sla'],       'platform/metrics/sla')
create_ns('platform-prod', ['incidents'],            'platform/incidents')
create_ns('platform-prod', ['incidents', 'alerts'],  'platform/incidents/alerts')
create_ns('platform-prod', ['governance'],           'platform/governance')
create_ns('platform-prod', ['governance', 'lineage'],'platform/governance/lineage')

print('\n✅ Step 6 complete')

=== Step 6: Create Namespaces ===

── raw-prod ──
  ⚠️  [409] raw-prod/web
  ⚠️  [409] raw-prod/web.clickstream
  ⚠️  [409] raw-prod/web.pageviews
  ⚠️  [409] raw-prod/mobile
  ⚠️  [409] raw-prod/mobile.events
  ⚠️  [409] raw-prod/mobile.crashes
  ⚠️  [409] raw-prod/payment
  ⚠️  [409] raw-prod/payment.transactions
  ⚠️  [409] raw-prod/payment.refunds
  ⚠️  [409] raw-prod/system
  ⚠️  [409] raw-prod/system.logs

── analytics-prod ──
  ⚠️  [403] analytics-prod/staging
  ⚠️  [404] analytics-prod/staging.web
  ⚠️  [404] analytics-prod/staging.payment
  ⚠️  [403] analytics-prod/mart
  ⚠️  [404] analytics-prod/mart.user
  ⚠️  [404] analytics-prod/mart.revenue
  ⚠️  [403] analytics-prod/reporting
  ⚠️  [404] analytics-prod/reporting.daily
  ⚠️  [404] analytics-prod/reporting.monthly

── ml-prod ──
  ⚠️  [409] ml-prod/features
  ⚠️  [409] ml-prod/features.user
  ⚠️  [409] ml-prod/features.product
  ⚠️  [409] ml-prod/models
  ⚠️  [409] ml-prod/models.experimental
  ⚠️  [409] ml-prod/models.pro

## Step 7 — Tables
각 네임스페이스에 테이블 생성.

In [64]:
print('=== Step 7: Create Tables ===')

def create_table(catalog, namespace, name, fields):
    ns = namespace if isinstance(namespace, list) else [namespace]
    ns_path = '%1F'.join(ns)  # Iceberg namespace separator
    r = requests.post(
        f'{CAT}/{catalog}/namespaces/{"%1F".join(ns)}/tables',
        headers=h(TOKEN),
        json={
            'name': name,
            'schema': {'type': 'struct', 'schema-id': 0, 'fields': fields},
            'partition-spec': {'spec-id': 0, 'fields': []},
            'write-order': {'order-id': 0, 'fields': []},
            'stage-create': False,
            'properties': {'write.format.default': 'parquet'}
        }
    )
    ns_str = '.'.join(ns)
    icon = '✅' if r.status_code in [200, 201] else '⚠️ '
    print(f'  {icon} [{r.status_code}] {catalog}.{ns_str}.{name}')
    if r.status_code not in [200, 201] and r.text:
        print(f'       {r.json().get("error", {}).get("message", r.text[:100])}')

f = lambda fid, fname, ftype, req=False: {'id': fid, 'name': fname, 'type': ftype, 'required': req}

# raw-prod tables
print('\n── raw-prod ──')
create_table('raw-prod', ['web', 'clickstream'], 'click_events', [
    f(1,'event_id','string',True), f(2,'user_id','long'), f(3,'session_id','string'),
    f(4,'url','string'), f(5,'element','string'), f(6,'ts','timestamp')])
create_table('raw-prod', ['web', 'pageviews'], 'pageview_events', [
    f(1,'event_id','string',True), f(2,'user_id','long'), f(3,'page_url','string'),
    f(4,'referrer','string'), f(5,'duration_ms','long'), f(6,'ts','timestamp')])
create_table('raw-prod', ['mobile', 'events'], 'app_events', [
    f(1,'event_id','string',True), f(2,'user_id','long'), f(3,'device_id','string'),
    f(4,'event_type','string'), f(5,'platform','string'), f(6,'ts','timestamp')])
create_table('raw-prod', ['mobile', 'crashes'], 'crash_reports', [
    f(1,'crash_id','string',True), f(2,'device_id','string'), f(3,'os_version','string'),
    f(4,'stack_trace','string'), f(5,'ts','timestamp')])
create_table('raw-prod', ['payment', 'transactions'], 'payment_events', [
    f(1,'txn_id','string',True), f(2,'user_id','long'), f(3,'amount','double'),
    f(4,'currency','string'), f(5,'status','string'), f(6,'ts','timestamp')])
create_table('raw-prod', ['payment', 'refunds'], 'refund_events', [
    f(1,'refund_id','string',True), f(2,'txn_id','string'), f(3,'amount','double'),
    f(4,'reason','string'), f(5,'ts','timestamp')])
create_table('raw-prod', ['system', 'logs'], 'server_logs', [
    f(1,'log_id','string',True), f(2,'service','string'), f(3,'level','string'),
    f(4,'message','string'), f(5,'host','string'), f(6,'ts','timestamp')])

# analytics-prod tables
print('\n── analytics-prod ──')
create_table('analytics-prod', ['staging', 'web'], 'web_events_staged', [
    f(1,'event_id','string',True), f(2,'user_id','long'), f(3,'event_type','string'),
    f(4,'properties','string'), f(5,'processed_at','timestamp')])
create_table('analytics-prod', ['staging', 'payment'], 'payment_staged', [
    f(1,'txn_id','string',True), f(2,'user_id','long'), f(3,'amount','double'),
    f(4,'status','string'), f(5,'processed_at','timestamp')])
create_table('analytics-prod', ['mart', 'user'], 'user_metrics', [
    f(1,'user_id','long',True), f(2,'total_sessions','long'), f(3,'total_events','long'),
    f(4,'last_active','timestamp'), f(5,'updated_at','timestamp')])
create_table('analytics-prod', ['mart', 'revenue'], 'revenue_metrics', [
    f(1,'date','date',True), f(2,'region','string'), f(3,'product','string'),
    f(4,'revenue','double'), f(5,'txn_count','long')])
create_table('analytics-prod', ['reporting', 'daily'], 'daily_kpis', [
    f(1,'date','date',True), f(2,'dau','long'), f(3,'revenue','double'),
    f(4,'conversion_rate','double'), f(5,'region','string')])
create_table('analytics-prod', ['reporting', 'monthly'], 'monthly_kpis', [
    f(1,'month','string',True), f(2,'mau','long'), f(3,'revenue','double'),
    f(4,'growth_rate','double')])

# ml-prod tables
print('\n── ml-prod ──')
create_table('ml-prod', ['features', 'user'], 'user_features', [
    f(1,'user_id','long',True), f(2,'feature_vector','string'), f(3,'version','string'),
    f(4,'updated_at','timestamp')])
create_table('ml-prod', ['features', 'product'], 'product_features', [
    f(1,'product_id','long',True), f(2,'feature_vector','string'), f(3,'updated_at','timestamp')])
create_table('ml-prod', ['models', 'experimental'], 'model_versions', [
    f(1,'model_id','string',True), f(2,'name','string'), f(3,'version','string'),
    f(4,'metrics','string'), f(5,'created_at','timestamp')])
create_table('ml-prod', ['models', 'production'], 'deployed_models', [
    f(1,'model_id','string',True), f(2,'name','string'), f(3,'version','string'),
    f(4,'deployed_at','timestamp'), f(5,'deployed_by','string')])
create_table('ml-prod', ['experiments', 'runs'], 'experiment_runs', [
    f(1,'run_id','string',True), f(2,'experiment_name','string'), f(3,'model_id','string'),
    f(4,'accuracy','double'), f(5,'loss','double'), f(6,'created_at','timestamp')])
create_table('ml-prod', ['monitoring', 'drift'], 'model_drift', [
    f(1,'model_id','string',True), f(2,'drift_score','double'), f(3,'feature_name','string'),
    f(4,'detected_at','timestamp')])

# finance-prod tables (PII marked in description)
print('\n── finance-prod ──')
create_table('finance-prod', ['ledger', 'accounts'], 'accounts', [
    f(1,'account_id','string',True), f(2,'account_name','string'), f(3,'account_type','string'),
    f(4,'card_number','string'),  # PII
    f(5,'balance','double'), f(6,'currency','string'), f(7,'updated_at','timestamp')])
create_table('finance-prod', ['ledger', 'entries'], 'journal_entries', [
    f(1,'entry_id','string',True), f(2,'account_id','string'), f(3,'amount','double'),
    f(4,'description','string'), f(5,'entry_date','date')])
create_table('finance-prod', ['reporting', 'pnl'], 'profit_loss', [
    f(1,'period','string',True), f(2,'region','string'), f(3,'revenue','double'),
    f(4,'expenses','double'), f(5,'net_profit','double')])
create_table('finance-prod', ['reporting', 'balance'], 'balance_sheet', [
    f(1,'period','string',True), f(2,'total_assets','double'), f(3,'total_liabilities','double'),
    f(4,'equity','double')])
create_table('finance-prod', ['audit', 'access'], 'access_log', [
    f(1,'log_id','string',True), f(2,'principal','string'), f(3,'resource','string'),
    f(4,'action','string'), f(5,'ts','timestamp')])
create_table('finance-prod', ['audit', 'changes'], 'change_log', [
    f(1,'change_id','string',True), f(2,'table_name','string'), f(3,'changed_by','string'),
    f(4,'before_value','string'), f(5,'after_value','string'), f(6,'ts','timestamp')])

# platform-prod tables
print('\n── platform-prod ──')
create_table('platform-prod', ['metrics', 'infra'], 'infra_metrics', [
    f(1,'host','string',True), f(2,'metric_name','string'), f(3,'value','double'),
    f(4,'unit','string'), f(5,'ts','timestamp')])
create_table('platform-prod', ['metrics', 'sla'], 'sla_metrics', [
    f(1,'service','string',True), f(2,'sla_target','double'), f(3,'actual','double'),
    f(4,'period','string'), f(5,'ts','timestamp')])
create_table('platform-prod', ['incidents', 'alerts'], 'alert_history', [
    f(1,'alert_id','string',True), f(2,'severity','string'), f(3,'service','string'),
    f(4,'message','string'), f(5,'resolved_at','timestamp'), f(6,'ts','timestamp')])
create_table('platform-prod', ['governance', 'lineage'], 'data_lineage', [
    f(1,'lineage_id','string',True), f(2,'source_table','string'), f(3,'target_table','string'),
    f(4,'transformation','string'), f(5,'recorded_at','timestamp')])

print('\n✅ Step 7 complete')

=== Step 7: Create Tables ===

── raw-prod ──
  ⚠️  [409] raw-prod.web.clickstream.click_events
       Table already exists: web.clickstream.click_events
  ⚠️  [409] raw-prod.web.pageviews.pageview_events
       Table already exists: web.pageviews.pageview_events
  ⚠️  [409] raw-prod.mobile.events.app_events
       Table already exists: mobile.events.app_events
  ⚠️  [409] raw-prod.mobile.crashes.crash_reports
       Table already exists: mobile.crashes.crash_reports
  ⚠️  [409] raw-prod.payment.transactions.payment_events
       Table already exists: payment.transactions.payment_events
  ⚠️  [409] raw-prod.payment.refunds.refund_events
       Table already exists: payment.refunds.refund_events
  ⚠️  [409] raw-prod.system.logs.server_logs
       Table already exists: system.logs.server_logs

── analytics-prod ──
  ⚠️  [404] analytics-prod.staging.web.web_events_staged
       Namespace does not exist: staging.web
  ⚠️  [404] analytics-prod.staging.payment.payment_staged
       Namespace

## Step 8 — Views
뷰 생성 — 보안 레이어 및 편의성 목적.
특히 finance-prod.ledger.accounts_masked는 PII 마스킹 뷰.

In [65]:
print('=== Step 8: Create Views ===')

def create_view(catalog, namespace, name, sql, fields):
    ns = namespace if isinstance(namespace, list) else [namespace]
    r = requests.post(
        f'{CAT}/{catalog}/namespaces/{"%1F".join(ns)}/views',
        headers=h(TOKEN),
        json={
            'name': name,
            'schema': {'type': 'struct', 'schema-id': 0, 'fields': fields},
            'view-version': {
                'version-id': 1, 'schema-id': 0, 'timestamp-ms': 0,
                'summary': {'engine-name': 'spark'},
                'default-namespace': ns,
                'representations': [{'type': 'sql', 'sql': sql, 'dialect': 'spark'}]
            },
            'properties': {}
        }
    )
    ns_str = '.'.join(ns)
    icon = '✅' if r.status_code in [200, 201] else '⚠️ '
    print(f'  {icon} [{r.status_code}] {catalog}.{ns_str}.{name}')
    if r.status_code not in [200, 201] and r.text:
        print(f'       {r.json().get("error", {}).get("message", r.text[:120])}')

f = lambda fid, fname, ftype, req=False: {'id': fid, 'name': fname, 'type': ftype, 'required': req}

# analytics-prod views
print('\n── analytics-prod ──')
create_view('analytics-prod', ['mart', 'user'], 'active_users_30d',
    'SELECT user_id, total_sessions, last_active FROM user_metrics WHERE last_active >= date_sub(current_date(), 30)',
    [f(1,'user_id','long',True), f(2,'total_sessions','long'), f(3,'last_active','timestamp')])

create_view('analytics-prod', ['mart', 'user'], 'churned_users',
    'SELECT user_id, last_active FROM user_metrics WHERE last_active < date_sub(current_date(), 90)',
    [f(1,'user_id','long',True), f(2,'last_active','timestamp')])

create_view('analytics-prod', ['mart', 'revenue'], 'revenue_by_region',
    'SELECT region, SUM(revenue) AS total_revenue, SUM(txn_count) AS total_txns FROM revenue_metrics GROUP BY region',
    [f(1,'region','string',True), f(2,'total_revenue','double'), f(3,'total_txns','long')])

create_view('analytics-prod', ['mart', 'revenue'], 'revenue_by_product',
    'SELECT product, SUM(revenue) AS total_revenue FROM revenue_metrics GROUP BY product',
    [f(1,'product','string',True), f(2,'total_revenue','double')])

create_view('analytics-prod', ['reporting', 'daily'], 'kpi_dashboard',
    'SELECT date, dau, revenue, conversion_rate FROM daily_kpis ORDER BY date DESC',
    [f(1,'date','date',True), f(2,'dau','long'), f(3,'revenue','double'), f(4,'conversion_rate','double')])

create_view('analytics-prod', ['reporting', 'monthly'], 'executive_summary',
    'SELECT month, mau, revenue, growth_rate FROM monthly_kpis ORDER BY month DESC',
    [f(1,'month','string',True), f(2,'mau','long'), f(3,'revenue','double'), f(4,'growth_rate','double')])

# ml-prod views
print('\n── ml-prod ──')
create_view('ml-prod', ['features', 'user'], 'feature_snapshot',
    'SELECT user_id, feature_vector, updated_at FROM user_features WHERE updated_at = (SELECT MAX(updated_at) FROM user_features)',
    [f(1,'user_id','long',True), f(2,'feature_vector','string'), f(3,'updated_at','timestamp')])

create_view('ml-prod', ['monitoring', 'drift'], 'model_health_summary',
    'SELECT model_id, AVG(drift_score) AS avg_drift, MAX(detected_at) AS last_detected FROM model_drift GROUP BY model_id',
    [f(1,'model_id','string',True), f(2,'avg_drift','double'), f(3,'last_detected','timestamp')])

# finance-prod views — PII masking layer
print('\n── finance-prod (PII masking) ──')
create_view('finance-prod', ['ledger', 'accounts'], 'accounts_masked',
    "SELECT account_id, account_name, account_type, CONCAT('****-****-****-', RIGHT(card_number, 4)) AS card_number_masked, CASE WHEN balance < 0 THEN 'negative' WHEN balance < 1000 THEN 'low' WHEN balance < 10000 THEN 'medium' ELSE 'high' END AS balance_range, currency FROM accounts",
    [f(1,'account_id','string',True), f(2,'account_name','string'), f(3,'account_type','string'),
     f(4,'card_number_masked','string'), f(5,'balance_range','string'), f(6,'currency','string')])

create_view('finance-prod', ['reporting', 'pnl'], 'pnl_summary',
    'SELECT period, SUM(revenue) AS total_revenue, SUM(expenses) AS total_expenses, SUM(net_profit) AS total_profit FROM profit_loss GROUP BY period',
    [f(1,'period','string',True), f(2,'total_revenue','double'), f(3,'total_expenses','double'), f(4,'total_profit','double')])

create_view('finance-prod', ['reporting', 'pnl'], 'pnl_by_region',
    'SELECT period, region, revenue, net_profit FROM profit_loss ORDER BY period DESC, net_profit DESC',
    [f(1,'period','string',True), f(2,'region','string'), f(3,'revenue','double'), f(4,'net_profit','double')])

create_view('finance-prod', ['audit', 'access'], 'audit_summary',
    'SELECT principal, COUNT(*) AS access_count, MAX(ts) AS last_access FROM access_log GROUP BY principal',
    [f(1,'principal','string',True), f(2,'access_count','long'), f(3,'last_access','timestamp')])

# platform-prod views
print('\n── platform-prod ──')
create_view('platform-prod', ['metrics', 'sla'], 'sla_violations',
    'SELECT service, sla_target, actual, period FROM sla_metrics WHERE actual < sla_target ORDER BY period DESC',
    [f(1,'service','string',True), f(2,'sla_target','double'), f(3,'actual','double'), f(4,'period','string')])

print('\n✅ Step 8 complete')

=== Step 8: Create Views ===

── analytics-prod ──
  ⚠️  [404] analytics-prod.mart.user.active_users_30d
       Namespace does not exist: mart.user
  ⚠️  [404] analytics-prod.mart.user.churned_users
       Namespace does not exist: mart.user
  ⚠️  [404] analytics-prod.mart.revenue.revenue_by_region
       Namespace does not exist: mart.revenue
  ⚠️  [404] analytics-prod.mart.revenue.revenue_by_product
       Namespace does not exist: mart.revenue
  ⚠️  [404] analytics-prod.reporting.daily.kpi_dashboard
       Namespace does not exist: reporting.daily
  ⚠️  [404] analytics-prod.reporting.monthly.executive_summary
       Namespace does not exist: reporting.monthly

── ml-prod ──
  ⚠️  [409] ml-prod.features.user.feature_snapshot
       View already exists: features.user.feature_snapshot
  ⚠️  [409] ml-prod.monitoring.drift.model_health_summary
       View already exists: monitoring.drift.model_health_summary

── finance-prod (PII masking) ──
  ⚠️  [409] finance-prod.ledger.accounts.accou

## Step 9 — Verification
생성된 전체 리소스 확인.

In [66]:
print('=== Step 9: Verification ===')

all_catalogs    = requests.get(f'{BASE}/catalogs',        headers=h(TOKEN)).json().get('catalogs', [])
all_principals  = requests.get(f'{BASE}/principals',      headers=h(TOKEN)).json().get('principals', [])
all_pr          = requests.get(f'{BASE}/principal-roles', headers=h(TOKEN)).json().get('roles', [])

print(f'\n📊 Summary:')
print(f'  Catalogs:        {len(all_catalogs)}')
print(f'  Principals:      {len([p for p in all_principals if p["name"] != "root"])}')
print(f'  Principal Roles: {len([r for r in all_pr if r["name"] != "service_admin"])}')

total_ns, total_tables, total_views, total_cr = 0, 0, 0, 0

print(f'\n📚 Catalog details:')
for c in all_catalogs:
    cname = c['name']
    cr_r = requests.get(f'{BASE}/catalogs/{cname}/catalog-roles', headers=h(TOKEN))
    cr_count = len(cr_r.json().get('roles', []))
    total_cr += cr_count

    # Fetch ALL namespaces including nested — use parent='' to get all
    ns_r = requests.get(
        f'{CAT}/{cname}/namespaces',
        headers=h(TOKEN),
        params={'parent': ''}  # empty parent = top level only
    )
    
    # Recursively get all namespaces
    def get_all_namespaces(catalog, parent=None):
        params = {}
        if parent:
            params['parent'] = '\x1f'.join(parent) if isinstance(parent, list) else parent
        r = requests.get(f'{CAT}/{catalog}/namespaces', headers=h(TOKEN), params=params)
        ns_list = r.json().get('namespaces', [])
        all_ns = []
        for ns in ns_list:
            ns_parts = ns if isinstance(ns, list) else [ns]
            all_ns.append(ns_parts)
            # Recurse into children
            children = get_all_namespaces(catalog, ns_parts)
            all_ns.extend(children)
        return all_ns

    all_ns = get_all_namespaces(cname)
    total_ns += len(all_ns)

    t_count, v_count = 0, 0
    for ns_parts in all_ns:
        ns_url = '%1F'.join(ns_parts)
        t_r = requests.get(f'{CAT}/{cname}/namespaces/{ns_url}/tables', headers=h(TOKEN))
        t_count += len(t_r.json().get('identifiers', []))
        v_r = requests.get(f'{CAT}/{cname}/namespaces/{ns_url}/views', headers=h(TOKEN))
        v_count += len(v_r.json().get('identifiers', []))

    total_tables += t_count
    total_views  += v_count
    print(f'  {cname}: {cr_count} catalog-roles, {len(all_ns)} namespaces, {t_count} tables, {v_count} views')

print(f'\n  Total catalog roles : {total_cr}')
print(f'  Total namespaces    : {total_ns}')
print(f'  Total tables        : {total_tables}')
print(f'  Total views         : {total_views}')

print(f'\n🔑 Key production patterns:')
print(f'  ✅ View as PII masking     : finance-prod.ledger.accounts → accounts_masked')
print(f'  ✅ Model promotion gate    : ml-engineer (experimental only) vs ml-lead (production write)')
print(f'  ✅ Service least privilege : spark-ingestion (web/mobile/payment write only)')
print(f'  ✅ Data steward pattern    : metadata only, no data rows')
print(f'  ✅ Audit service           : READ_PROPERTIES only across all catalogs')
print(f'  ✅ Finance isolation       : finance-analyst → reporting views only, no ledger access')

=== Step 9: Verification ===

📊 Summary:
  Catalogs:        5
  Principals:      12
  Principal Roles: 10

📚 Catalog details:
  raw-prod: 7 catalog-roles, 11 namespaces, 7 tables, 0 views
  analytics-prod: 0 catalog-roles, 0 namespaces, 0 tables, 0 views
  ml-prod: 4 catalog-roles, 10 namespaces, 6 tables, 2 views
  finance-prod: 4 catalog-roles, 9 namespaces, 6 tables, 4 views
  platform-prod: 4 catalog-roles, 7 namespaces, 4 tables, 1 views

  Total catalog roles : 19
  Total namespaces    : 37
  Total tables        : 23
  Total views         : 7

🔑 Key production patterns:
  ✅ View as PII masking     : finance-prod.ledger.accounts → accounts_masked
  ✅ Model promotion gate    : ml-engineer (experimental only) vs ml-lead (production write)
  ✅ Service least privilege : spark-ingestion (web/mobile/payment write only)
  ✅ Data steward pattern    : metadata only, no data rows
  ✅ Audit service           : READ_PROPERTIES only across all catalogs
  ✅ Finance isolation       : finance-ana

In [67]:
# Show grants with scope breakdown
print('=== Grant Scope Analysis ===')

for c in all_catalogs:
    cname = c['name']
    cr_list = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles', headers=h(TOKEN)).json().get('roles', [])
    print(f'\nCatalog: {cname}')
    for cr in cr_list:
        crname = cr['name']
        grants = requests.get(f'{POLARIS_URL}/api/management/v1/catalogs/{cname}/catalog-roles/{crname}/grants', headers=h(TOKEN)).json().get('grants', [])
        if not grants:
            continue
        print(f'  Catalog Role: {crname}')
        catalog_grants   = [g for g in grants if g['type'] == 'catalog']
        namespace_grants = [g for g in grants if g['type'] == 'namespace']
        table_grants     = [g for g in grants if g['type'] == 'table']
        view_grants      = [g for g in grants if g['type'] == 'view']
        if catalog_grants:
            print(f'    [catalog scope]   → applies to entire catalog')
            for g in catalog_grants:
                print(f'      {g["privilege"]}')
        if namespace_grants:
            print(f'    [namespace scope] → applies to specific namespace only')
            for g in namespace_grants:
                print(f'      {g["privilege"]} on namespace:{g["namespace"]}')
        if table_grants:
            print(f'    [table scope]     → applies to specific table only')
            for g in table_grants:
                print(f'      {g["privilege"]} on {g["namespace"]}.{g["tableName"]}')
        if view_grants:
            print(f'    [view scope]      → applies to specific view only')
            for g in view_grants:
                print(f'      {g["privilege"]} on {g["namespace"]}.{g["viewName"]}')

=== Grant Scope Analysis ===

Catalog: raw-prod
  Catalog Role: catalog_admin
    [catalog scope]   → applies to entire catalog
      CATALOG_MANAGE_ACCESS
      CATALOG_MANAGE_METADATA
  Catalog Role: meta-role
    [catalog scope]   → applies to entire catalog
      CATALOG_MANAGE_METADATA
      NAMESPACE_LIST
  Catalog Role: prop-role
    [catalog scope]   → applies to entire catalog
      CATALOG_READ_PROPERTIES
  Catalog Role: ro-role
    [catalog scope]   → applies to entire catalog
      TABLE_READ_DATA
      NAMESPACE_LIST
      CATALOG_READ_PROPERTIES
  Catalog Role: rw-role
    [catalog scope]   → applies to entire catalog
      CATALOG_MANAGE_CONTENT
      CATALOG_READ_PROPERTIES
  Catalog Role: write-payment-role
    [catalog scope]   → applies to entire catalog
      TABLE_WRITE_DATA
    [namespace scope] → applies to specific namespace only
      NAMESPACE_FULL_METADATA on namespace:['payment']
  Catalog Role: write-web-role
    [catalog scope]   → applies to entire catalo

## 관찰 결과 (Observations)

- 5개 카탈로그가 완전히 격리된 도메인으로 구성
- 중첩 네임스페이스 (web/clickstream, models/production 등)로 계층적 데이터 구조 표현
- View as masking: finance-prod.ledger.accounts 직접 접근 불가, accounts_masked만 노출
- ml-lead만 models.production에 쓰기 가능 — 모델 프로모션 게이트
- 서비스 계정별 최소 권한 — spark-ingestion은 raw-prod 특정 네임스페이스만
- security-auditor/audit-service: 데이터 접근 없이 카탈로그 존재 여부만 확인 가능